In [ ]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from spikingjelly.activation_based.surrogate import ATan, Sigmoid

# Add the current directory to sys.path to import from src
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from src.noisy_node import NoisyIFNode

In [ ]:
# Cell: Helper Function
def plot_noisy_approximation(node, v_mem_range, title, k_trials=1000):
    """
    Simulates the NoisyNode and plots its firing probability against the surrogate's primitive function.
    """
    # 1. Simulate Noisy Node
    # Create input tensor: (k_trials, len(v_mem_range))
    v_input = torch.tensor(v_mem_range, dtype=torch.float32).repeat(k_trials, 1)
    
    # Set node state directly
    node.v = v_input
    
    # Fire! (This uses the noisy mechanism)
    with torch.no_grad():
        spikes = node.neuronal_fire()
    
    # Calculate probability
    prob_simulated = spikes.mean(dim=0).numpy()
    
    # 2. Calculate Theoretical Value (Surrogate Primitive)
    # The surrogate function takes (input - threshold)
    # We temporarily switch to non-spiking mode to get the primitive value
    original_mode = node.surrogate_function.spiking
    node.surrogate_function.set_spiking_mode(False)
    
    v_tensor = torch.tensor(v_mem_range, dtype=torch.float32)
    with torch.no_grad():
        # Note: surrogate function input is (v - v_threshold)
        prob_theoretical = node.surrogate_function(v_tensor - node.v_threshold).numpy()
        
    node.surrogate_function.set_spiking_mode(original_mode)
    
    # 3. Plot
    plt.figure(figsize=(10, 6))
    plt.plot(v_mem_range, prob_simulated, label='Simulated Noisy Node', color='#d62728', linewidth=3, alpha=0.8)
    plt.plot(v_mem_range, prob_theoretical, label='Surrogate Primitive (Expected)', color='blue', linestyle='--', linewidth=2)
    
    # Add threshold line
    plt.axvline(x=node.v_threshold, color='gray', linestyle=':', alpha=0.6, label='$V_{thr}$')
    
    plt.title(title, fontsize=14)
    plt.xlabel('Membrane Potential ($V_{mem}$)', fontsize=12)
    plt.ylabel('Firing Probability / Output', fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell: Sigmoid Noise Experiment
alpha_sigmoid = 4.0
surrogate_sigmoid = Sigmoid(alpha=alpha_sigmoid, spiking=True)
node_sigmoid = NoisyIFNode(noise_type='surrogate', surrogate_function=surrogate_sigmoid, v_threshold=1.0, v_reset=0.0)

v_mem = np.linspace(-1, 3, 1000)
plot_noisy_approximation(node_sigmoid, v_mem, f'Sigmoid Noise Approximation (alpha={alpha_sigmoid})')

In [ ]:
# Cell: ATan Noise Experiment
alpha_atan = 2.0
surrogate_atan = ATan(alpha=alpha_atan, spiking=True)
node_atan = NoisyIFNode(noise_type='surrogate', surrogate_function=surrogate_atan, v_threshold=1.0, v_reset=0.0)

v_mem = np.linspace(-2, 4, 1000)
plot_noisy_approximation(node_atan, v_mem, f'ATan Noise Approximation (alpha={alpha_atan})')

In [ ]:
# For uniform noise, the theoretical approximation is a linear ramp (clamped).
node_uniform = NoisyIFNode(noise_type='uniform', v_threshold=1.0, v_reset=0.0)

v_mem = np.linspace(-0.5, 2.5, 1000)
v_input = torch.tensor(v_mem, dtype=torch.float32).repeat(5000, 1)
node_uniform.v = v_input
with torch.no_grad():
    spikes = node_uniform.neuronal_fire()
prob_simulated = spikes.mean(dim=0).numpy()

# Theoretical Ramp for U(0, 1) noise and V_thr=1
prob_theoretical = np.clip(v_mem, 0, 1)

plt.figure(figsize=(10, 6))
plt.plot(v_mem, prob_simulated, label='Simulated Uniform Noise', color='#d62728', linewidth=3, alpha=0.8)
plt.plot(v_mem, prob_theoretical, label='Theoretical Ramp (Linear)', color='blue', linestyle='--', linewidth=2)
plt.axvline(x=1.0, color='gray', linestyle=':', alpha=0.6, label='$V_{thr}$')
plt.title('Uniform Noise Approximation', fontsize=14)
plt.xlabel('Membrane Potential ($V_{mem}$)', fontsize=12)
plt.ylabel('Firing Probability', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from spikingjelly.activation_based import surrogate, neuron
# --- 2. The Numerical Argument Experiment ---

def run_alignment_experiment(surrogate_obj, noise_variants, n_trials=1000):
    v_th = 1.0
    v_range = torch.linspace(v_th - 3.0, v_th + 3.0, 100)
    
    # Theoretical Ground Truth
    # Use the surrogate function's primitive directly
    original_mode = surrogate_obj.spiking
    surrogate_obj.set_spiking_mode(False)
    
    with torch.no_grad():
        y_ground_truth = surrogate_obj(v_range - v_th)
        
    surrogate_obj.set_spiking_mode(original_mode)

    if isinstance(surrogate_obj, surrogate.Sigmoid):
        s_name = "Sigmoid"
    elif isinstance(surrogate_obj, surrogate.ATan):
        s_name = "ATan"
    else:
        s_name = surrogate_obj.__class__.__name__
    results = {}
    mse_scores = {}
    
    for noise_name, noise_type_key in noise_variants.items():
        # 1. Instantiate the actual node class you use in your model
        # We use a dummy threshold/decay because we will bypass integration anyway
        node = NoisyIFNode(
            noise_type=noise_type_key, 
            surrogate_function=surrogate_obj, 
            v_threshold=v_th,
            detach_reset=True
        )
        
        fire_sum = torch.zeros_like(v_range)
        
        with torch.no_grad():
            for _ in range(n_trials):
                node.v = v_range 
                fires = node.neuronal_fire()
                
                fire_sum += fires.float() # Ensure float for averaging
        
        y_empirical = fire_sum / n_trials
        results[noise_name] = y_empirical
        mse_scores[noise_name] = torch.mean((y_ground_truth - y_empirical)**2).item()

    return v_range, y_ground_truth, results, mse_scores, s_name

# --- 3. Execute & Plot ---

# Scenario A: Network trained with ATan Surrogate (Common in SpikingJelly)

noise_candidates = {
    "Uniform Noise": "uniform",
    "Gaussian Noise": "gaussian", 
    "Matched Noise": "surrogate" # This uses your _create_atan_noise
}

results = [run_alignment_experiment(surr_func, noise_candidates) for surr_func in [surrogate.Sigmoid(alpha=4.0), surrogate.ATan(alpha=2.0)]]

for v_inputs, y_true, y_results, mse, s_name in results:
    # Plotting
    plt.figure(figsize=(10, 6))
    plt.plot(v_inputs, y_true, 'k--', linewidth=3, label=f'Theoretical {s_name} (Ground Truth)')

    colors = {'Uniform Noise': 'red', 'Gaussian Noise': 'orange', 'Matched (Cauchy)': 'green'}
    for name, y_curve in y_results.items():
        plt.plot(v_inputs, y_curve, linewidth=2, color=colors.get(name, 'blue'), 
                label=f'{name} (MSE: {mse[name]:.5f})')

    plt.title(f"Noise Alignment for {s_name} Surrogate")
    plt.xlabel("Membrane Potential (v)")
    plt.ylabel("Firing Probability P(fire)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    # --- 4. Generate The Argument Table ---
    df = pd.DataFrame(list(mse.items()), columns=["Noise Strategy", "MSE (Lower is Better)"])
    df = df.sort_values("MSE (Lower is Better)")
    print("\n quantitative JUSTIFICATION:")
    print(df.to_string(index=False))